# Progress Report 1 — Week 1-2
## Transformers in Financial Derivative Pricing: Solving the Black-Scholes Equation using PINNsFormer
**Ceyhan Doğan — CMP712 — Hacettepe University**

---

### Contents
1. Problem Definition & Motivation
2. Literature Review Summary
3. Domain Transfer: Physical → Financial
4. Synthetic Dataset Generation (Call & Put)
5. Baseline: Standard PINN vs Analytical Solution
6. Results
7. Summary & Next Steps

---
## 1. Problem Definition & Motivation

The **Black-Scholes PDE** governs European option pricing:

$$\frac{\partial V}{\partial t} + \frac{1}{2}\sigma^2 S^2 \frac{\partial^2 V}{\partial S^2} + rS\frac{\partial V}{\partial S} - rV = 0$$

where $V$ is the option value, $S$ the underlying asset price, $t$ the time, $r$ the risk-free rate, and $\sigma$ the volatility.

**Why PINNs?**  
Traditional solvers (finite element, finite difference) are expensive when rerunning across parameter sweeps. Physics-Informed Neural Networks (PINNs) embed the PDE directly into the loss function and solve it without labeled data.

**Why PINNsFormer?**  
Vanilla PINNs treat each collocation point independently (*pointwise* training), which causes **spectral bias** — failure to capture high-frequency features near the strike price. PINNsFormer (Zhao et al., 2024) replaces pointwise training with a *sequential* loss via a Transformer encoder-decoder and uses WaveAct activation ($w_1 \sin(x) + w_2 \cos(x)$) to address spectral bias.

---
## 2. Literature Review Summary

| Reference | Contribution | Limitation |
|-----------|-------------|------------|
| Raissi et al. (2019) | Foundational PINN framework — PDE embedded in loss | Pointwise; spectral bias |
| Dhiman & Hu (2023) | PINN for BS call/put; ~30% gain over analytic benchmarks | Convergence issues near boundaries |
| Nuugulu et al. (2025) | PINN vs pure data-driven on BS & Heston; PINN superior in stability | Both still pointwise |
| **Zhao et al. (2024)** | **PINNsFormer**: Transformer + WaveAct + sequential loss → mitigates spectral bias | Not applied to finance |

**This project** bridges the gap: adapts PINNsFormer to the Black-Scholes domain.

---
## 3. Domain Transfer: Physical → Financial

PINNsFormer was designed for physics PDEs with inputs $(x, t)$. We map:

| PINNsFormer | Black-Scholes | Notes |
|-------------|--------------|-------|
| $x$ (space) | $S$ (asset price) | $S \in [20, 200]$ |
| $t$ (time)  | $\tau = T - t$ (time-to-maturity) | Reversed; $\tau=0$ is terminal |
| $u(x,t)$    | $V(S,\tau)$ (option price) | |

Working with $\tau$ instead of $t$ converts the terminal condition $V(S,T)=\text{payoff}$ into an **initial condition** $V(S,0)=\text{payoff}$, which is the natural form for PINNs.

**PDE in $(S, \tau)$ notation:**
$$\frac{\partial V}{\partial \tau} - \frac{1}{2}\sigma^2 S^2 \frac{\partial^2 V}{\partial S^2} - rS\frac{\partial V}{\partial S} + rV = 0$$

**Boundary & terminal conditions (European call):**

| Condition | Expression |
|-----------|------------|
| Terminal ($\tau=0$) | $V(S, 0) = \max(S - K, 0)$ |
| Left BC ($S \to 0$) | $V(S_{\min}, \tau) \approx 0$ |
| Right BC ($S \to \infty$) | $V(S_{\max}, \tau) \approx S_{\max} - K e^{-r\tau}$ |

---
## 4. Synthetic Dataset Generation

In [1]:
import sys, os
sys.path.insert(0, 'bs_pinn')
sys.path.insert(0, 'pinnsformer')

import numpy as np
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

from black_scholes_data import (
    generate_dataset, bs_call, bs_put,
    sample_residual_points, sample_terminal_points, sample_boundary_points
)
from bs_pinn_model import BS_PINN, rMAE, rRMSE

# ── Parameters ────────────────────────────────────────────────────────────────
K, r, sigma   = 100.0, 0.05, 0.20
S_MIN, S_MAX  = 20.0, 200.0
TAU_MAX       = 1.0
DEVICE        = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

# ── Datasets ──────────────────────────────────────────────────────────────────
ds_call = generate_dataset(K=K, r=r, sigma=sigma, n_S=100, n_tau=100, option_type='call')
ds_put  = generate_dataset(K=K, r=r, sigma=sigma, n_S=100, n_tau=100, option_type='put')

print(f'Call price grid shape : {ds_call["V_mesh"].shape}')
print(f'Put  price grid shape : {ds_put["V_mesh"].shape}')
print(f'Sample call price at S=100, τ=0.5 : {bs_call(100, K, r, sigma, 0.5):.4f}')
print(f'Sample put  price at S=100, τ=0.5 : {bs_put(100, K, r, sigma, 0.5):.4f}')

Device: cpu
Call price grid shape : (100, 100)
Put  price grid shape : (100, 100)
Sample call price at S=100, τ=0.5 : 6.8887
Sample put  price at S=100, τ=0.5 : 4.4197


In [2]:
# ── Price surfaces ────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(14, 5))
for idx, (ds, title) in enumerate([(ds_call, 'European Call'), (ds_put, 'European Put')]):
    ax = fig.add_subplot(1, 2, idx+1, projection='3d')
    ax.plot_surface(ds['S_mesh'], ds['tau_mesh'], ds['V_mesh'], cmap='viridis', alpha=0.85)
    ax.set_xlabel('Asset Price S'); ax.set_ylabel('Time-to-Maturity τ'); ax.set_zlabel('Option Price V')
    ax.set_title(f'Analytical B-S: {title}\n(K={K}, r={r}, σ={sigma})')
plt.tight_layout()
plt.savefig('bs_pinn/fig_price_surfaces.png', dpi=120)
plt.show()
print('Price surfaces saved.')

Price surfaces saved.


/tmp/ipykernel_37363/846016733.py:10: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


In [3]:
# ── Put-call parity check: C - P = S - K·e^{-rτ} ─────────────────────────────
parity_lhs = ds_call['V_mesh'] - ds_put['V_mesh']
parity_rhs = ds_call['S_mesh'] - K * np.exp(-r * ds_call['tau_mesh'])
mask = ds_call['tau_mesh'] > 0
max_err = np.max(np.abs(parity_lhs[mask] - parity_rhs[mask]))
print(f'Put-call parity max error: {max_err:.2e}  (should be ~machine epsilon)')

Put-call parity max error: 1.42e-14  (should be ~machine epsilon)


---
## 5. Baseline: Standard PINN vs Analytical Solution

We train a 6-layer tanh PINN with the loss:

$$\mathcal{L} = \lambda_1 \mathcal{L}_{\text{PDE}} + \lambda_2 \mathcal{L}_{\text{TC}} + \lambda_3 \mathcal{L}_{\text{BC}}$$

with $\lambda_1=1,\ \lambda_2=10,\ \lambda_3=10$ and collocation sizes $N_{\text{res}}=2000,\ N_{\text{TC}}=500,\ N_{\text{BC}}=500$.

In [4]:
import time

# ── Collocation points (fixed seeds for reproducibility) ─────────────────────
N_RES, N_TC, N_BC = 2000, 500, 500

S_res,  tau_res         = sample_residual_points(N_RES, S_MIN, S_MAX, 1e-3, TAU_MAX, seed=0)
S_tc,   tau_tc,  V_tc  = sample_terminal_points(N_TC, S_MIN, S_MAX, K, r, sigma, seed=1)
(S_bl, tau_bl, V_bl), (S_br, tau_br, V_br) = sample_boundary_points(
    N_BC, 1e-3, TAU_MAX, S_MIN, S_MAX, K, r, sigma, seed=2
)
TRAIN_ARGS = (S_res, tau_res, S_tc, tau_tc, V_tc, S_bl, tau_bl, V_bl, S_br, tau_br, V_br)

print(f'Residual points  : {N_RES}')
print(f'Terminal points  : {N_TC}')
print(f'Boundary points  : {N_BC} (per side)')

Residual points  : 2000
Terminal points  : 500
Boundary points  : 500 (per side)


In [5]:
# ── PINN training ─────────────────────────────────────────────────────────────
torch.manual_seed(SEED)

pinn = BS_PINN(hidden_dim=64, num_layer=6, sigma=sigma, r=r,
               lam1=1.0, lam2=10.0, lam3=10.0, lr=1e-3, device=DEVICE)

N_EPOCHS = 10000
history  = []
t0 = time.time()

for ep in range(1, N_EPOCHS + 1):
    losses = pinn.train_step(*TRAIN_ARGS)
    history.append(losses)
    if ep % 1000 == 0:
        print(f'Epoch {ep:5d} | total={losses[0]:.4e} | '
              f'L_pde={losses[1]:.4e} | L_tc={losses[2]:.4e} | L_bc={losses[3]:.4e}')

elapsed = time.time() - t0
print(f'\nTraining complete — {elapsed:.0f}s ({elapsed/60:.1f} min)')

Epoch  1000 | total=1.9167e+04 | L_pde=1.7287e+01 | L_tc=1.1425e+02 | L_bc=1.8007e+03


Epoch  2000 | total=1.6488e+03 | L_pde=8.9141e+01 | L_tc=2.4563e+00 | L_bc=1.5351e+02


Epoch  3000 | total=2.4915e+02 | L_pde=1.0656e+02 | L_tc=1.0020e+01 | L_bc=4.2389e+00


Epoch  4000 | total=2.6517e+01 | L_pde=4.5226e+00 | L_tc=4.0063e-01 | L_bc=1.7988e+00


Epoch  5000 | total=1.1937e+01 | L_pde=5.3954e+00 | L_tc=2.2829e-01 | L_bc=4.2587e-01


Epoch  6000 | total=3.1712e+00 | L_pde=1.4699e+00 | L_tc=1.0573e-01 | L_bc=6.4406e-02


Epoch  7000 | total=1.5515e+00 | L_pde=6.1045e-01 | L_tc=7.5641e-02 | L_bc=1.8468e-02


Epoch  8000 | total=3.1340e+00 | L_pde=4.4511e-01 | L_tc=1.1592e-01 | L_bc=1.5297e-01


Epoch  9000 | total=1.2048e+01 | L_pde=2.0204e+00 | L_tc=4.5463e-01 | L_bc=5.4810e-01


Epoch 10000 | total=1.2032e+00 | L_pde=3.6781e-01 | L_tc=5.5098e-02 | L_bc=2.8444e-02

Training complete — 143s (2.4 min)


---
## 6. Results

In [6]:
# ── Loss curves ───────────────────────────────────────────────────────────────
history = np.array(history)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].semilogy(history[:, 0], label='Total',  linewidth=1.5)
axes[0].semilogy(history[:, 1], label='L_pde',  linestyle='--', linewidth=1)
axes[0].semilogy(history[:, 2], label='L_tc',   linestyle=':',  linewidth=1)
axes[0].semilogy(history[:, 3], label='L_bc',   linestyle='-.', linewidth=1)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss (log)')
axes[0].set_title(f'PINN Training Loss — {N_EPOCHS} Epochs (seed={SEED})')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

half = N_EPOCHS // 2
axes[1].semilogy(range(half, N_EPOCHS), history[half:, 0], linewidth=1.5)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Total Loss (log)')
axes[1].set_title('Second Half (Convergence)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('bs_pinn/fig_pinn_loss.png', dpi=120)
plt.show()

/tmp/ipykernel_37363/2560431549.py:21: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


In [7]:
# ── Evaluation ────────────────────────────────────────────────────────────────
S_flat   = ds_call['flat']['S']
tau_flat = ds_call['flat']['tau']
V_true   = ds_call['flat']['V']
mask_eval = tau_flat > 1e-4

V_pred = pinn.predict(S_flat, tau_flat)
rmae   = rMAE(V_pred[mask_eval],  V_true[mask_eval])
rrmse  = rRMSE(V_pred[mask_eval], V_true[mask_eval])

print('='*45)
print(f'  Baseline PINN — European Call (10 000 ep)')
print('='*45)
print(f'  rMAE  = {rmae*100:.2f}%')
print(f'  rRMSE = {rrmse*100:.2f}%')
print(f'  Time  = {elapsed:.0f}s')
print('='*45)

  Baseline PINN — European Call (10 000 ep)
  rMAE  = 0.45%
  rRMSE = 0.35%
  Time  = 143s


In [8]:
# ── Price surface comparison ──────────────────────────────────────────────────
V_pred_grid  = V_pred.reshape(ds_call['V_mesh'].shape)
abs_err_grid = np.abs(V_pred_grid - ds_call['V_mesh'])

fig, axes = plt.subplots(1, 3, figsize=(18, 5), subplot_kw={'projection': '3d'})
for ax, Z, title, cmap in zip(
    axes,
    [ds_call['V_mesh'], V_pred_grid, abs_err_grid],
    ['Analytical B-S', 'PINN Prediction', '|Error|'],
    ['viridis', 'plasma', 'hot']
):
    ax.plot_surface(ds_call['S_mesh'], ds_call['tau_mesh'], Z, cmap=cmap, alpha=0.85)
    ax.set_xlabel('S'); ax.set_ylabel('τ'); ax.set_zlabel('V'); ax.set_title(title)

plt.suptitle(f'European Call — PINN vs Analytical B-S  (rMAE={rmae*100:.2f}%)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('bs_pinn/fig_pinn_vs_analytical.png', dpi=120)
plt.show()

/tmp/ipykernel_37363/3906057114.py:18: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


In [9]:
# ── Absolute error heatmap ────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
im = ax.contourf(ds_call['S_mesh'], ds_call['tau_mesh'], abs_err_grid, levels=20, cmap='hot_r')
plt.colorbar(im, ax=ax, label='|Error|')
ax.axvline(K, color='cyan', linewidth=1.5, linestyle='--', label=f'Strike K={K}')
ax.set_xlabel('Asset Price S'); ax.set_ylabel('Time-to-Maturity τ')
ax.set_title(f'PINN Absolute Error Heatmap  (rMAE={rmae*100:.2f}%)')
ax.legend()
plt.tight_layout()
plt.savefig('bs_pinn/fig_pinn_heatmap.png', dpi=120)
plt.show()

/tmp/ipykernel_37363/3030513153.py:11: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


In [10]:
# ── Slice comparison at fixed τ values ───────────────────────────────────────
tau_vals = [0.1, 0.3, 0.5, 1.0]
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for ax, tv in zip(axes, tau_vals):
    tau_arr = np.full_like(ds_call['S_grid'], tv)
    V_ana   = bs_call(ds_call['S_grid'], K, r, sigma, tv)
    V_nn    = pinn.predict(ds_call['S_grid'], tau_arr)
    ax.plot(ds_call['S_grid'], V_ana, 'k-',  label='Analytical', linewidth=2.5)
    ax.plot(ds_call['S_grid'], V_nn,  'r--', label='PINN',       linewidth=1.5)
    ax.axvline(K, color='gray', linestyle=':', alpha=0.5)
    ax.set_xlabel('S'); ax.set_ylabel('V'); ax.set_title(f'τ = {tv} yr')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.suptitle(f'European Call: Analytical vs PINN at Fixed τ  (seed={SEED}, N_res={N_RES})',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('bs_pinn/fig_pinn_slices.png', dpi=120)
plt.show()

/tmp/ipykernel_37363/197565278.py:19: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


---
## 7. Summary & Next Steps

### Accomplished (Week 1–2)

| Task | Status |
|------|--------|
| PINNsFormer repo klonlandı, orijinal modeller doğrulandı | ✅ |
| Domain transfer $(x,t) \to (S,\tau)$ implemente edildi | ✅ |
| Sentetik call & put veri setleri analitik B-S ile oluşturuldu | ✅ |
| Put-call parity doğrulandı (maks. hata ~$10^{-14}$) | ✅ |
| Baseline PINN eğitildi ve değerlendirildi | ✅ |
| Fiyat yüzeyi & dilim karşılaştırmaları çizildi | ✅ |

### Baseline PINN Sonuçları

| Ayar | Değer |
|------|-------|
| Mimari | 6 katman, 64 gizli birim, tanh |
| Epoch | 10 000 |
| N_res / N_TC / N_BC | 2 000 / 500 / 500 |
| λ₁ / λ₂ / λ₃ | 1 / 10 / 10 |
| **rMAE** | **0.45%** |
| **rRMSE** | **0.35%** |
| Eğitim süresi | ~151s |

### Gözlemler
- PINN genel call fiyat yüzeyini başarıyla yaklaşıyor.
- En büyük hatalar **strike fiyatı civarında** ($S \approx K$) ve **kısa vadelerde** — payoff'un kink içerdiği yüksek frekanslı bölge.
- Bu durum literatürde tanımlanan **spectral bias** sorununu doğrulamakta ve PINNsFormer'ın WaveAct + sequential loss yaklaşımını motive etmektedir.

### Hafta 3–4 Planı
1. **BS_PINNsFormer** (Finite Difference & Autograd versiyonları) eğitimi ve PINN ile karşılaştırması.
2. Hiperparametre taraması: $\lambda_1, \lambda_2, \lambda_3$, $d_{\text{model}}$, Transformer katman sayısı.
3. rMAE ve rRMSE metrikleriyle kapsamlı karşılaştırma.
4. Eğitim süresi benchmarkı: PINN vs PINNsFormer-FD vs PINNsFormer-AD.